# NHA Hackathon – Problem Statement 01  
## Clinical Document Classification & Compliance to STG requirements

This notebook is a **starter scaffold** for participants working on:

- mixed-quality healthcare document ingestion
- OCR + layout understanding
- visual cue detection
- STG / policy rule checks
- explainable claim decisioning
- episode timeline construction
- extra / non-required document identification


### Deliverables this notebook is designed to help produce
1. **Per-page/package JSON output** in the exact format required by the problem statement  
2. **Human-readable summary table** with document type, rule checks, and reasons  
3. **Episode timeline** with admission / investigation / procedure / discharge ordering  
4. **Decision**: `PASS`, `CONDITIONAL`, or `FAIL` with evidence and confidence

This notebook is a skeleton and can be changed by participants 

In [1]:
# =========================
# 1. INSTALLS / IMPORTS
# =========================

# Uncomment and adapt as needed during hackathon usage.
# !pip install pymupdf pdf2image pillow opencv-python pandas numpy pydantic python-dateutil rapidfuzz

from __future__ import annotations

import os
import re
import json
import math
import uuid
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from dataclasses import dataclass, field, asdict
from collections import defaultdict

import pandas as pd
import numpy as np

## Download the Dataset
We have provided a dedicated widget to download the hackathon datasets directly from the platform into this notebook environment.

### 1. Import the Widget

from databank_download_widget import DatabankDownloadWidget

### 2. Download the Databank
Select the cell below and run it.
Enter the Databank ID for the hackathon package.
Enter your email and password for the platform.
Click the **Download** button.

The widget will download and unzip the data right into your current directory. You can monitor the progress in the status output area below the button.

### **Databank ID for PS1: c110a5f8-6e79-43bd-bd7a-979677354958**

In [2]:
from databank_download_widget import DatabankDownloadWidget

# Create an instance of the databank widget
databank_downloader = DatabankDownloadWidget()

# Display the widget
databank_downloader.display()

In [3]:
# =========================
# 2. CONFIG
# =========================

DATA_ROOT = Path("./c110a5f8-6e79-43bd-bd7a-979677354958/Claims")         # input claims folder
OUTPUT_ROOT = Path("./outputs")    # json/csv/html outputs
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

SUPPORTED_EXTENSIONS = {".pdf", ".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

PACKAGE_CODES = ["MG064A", "SG039C", "MG006A", "SB039A"]

DECISION_PASS = "PASS"
DECISION_CONDITIONAL = "CONDITIONAL"
DECISION_FAIL = "FAIL"

In [4]:
import os 
os.listdir(DATA_ROOT)

['MG064A', '.DS_Store', 'SB039A', 'SG039C', 'MG006A']

### Example Usage of NHAclient LLM Call

Supports following models:

- Ministral 3B
- Ministral 8B
- Nemotron Nano 30B### Example Usage of NHAclient LLM Call

Supports following models:

- Ministral 3B
- Ministral 8B
- Nemotron Nano 30B
- Gemma 3 12B
- Gemma 3 4B

Can be used by Participants anywhere
- Gemma 3 12B
- Gemma 3 4B

Can be used by Participants anywhere

In [5]:
from nha_client import NHAclient
import base64


clientId="cd5a1414-36be-4193-907d-23795abb650b"
clientSecret="3fd366e6e8ae509f1ea1b562004191dd2d4b45a3"


nc = NHAclient(clientId, clientSecret)


with open("c110a5f8-6e79-43bd-bd7a-979677354958/Claims/MG006A/CMJAY_TR_CMJAY_2025_R3_1022010623/000590__CMJAY_TR_CMJAY_2025_R3_1022010623__36acf382-6069-49c4-b705-a1c62a644a67.jpg", "rb") as image_file:
    image_bytes = image_file.read()

image_base64 = base64.b64encode(image_bytes).decode("utf-8")
data_url = f"data:image/jpeg;base64,{image_base64}"

response = nc.completion(
    model="Gemma 3 4B", #use one of the mode
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": data_url}},
                {"type": "text", "text": "What do you see"},
            ],
        }
    ],
    metadata={
            "problem_statement":1
        }
)

print(response)

2026-04-29 19:42:31,281 - INFO - Fetching new auth token...
2026-04-29 19:42:31,359 - INFO - HTTP Request: POST https://aaehackathon.nhaad.in/controlplane/iudx/v2/auth/token "HTTP/1.1 200 OK"
2026-04-29 19:42:31,360 - INFO - Auth token obtained successfully
2026-04-29 19:42:33,704 - INFO - HTTP Request: POST https://aaehackathon.nhaad.in/nhm-proxy/v1/chat/completions "HTTP/1.1 200 OK"


{'id': 'chatcmpl-e6fdc262-b4f0-4830-9fe1-34d6a4cc9a75', 'created': 1777491753, 'model': 'google.gemma-3-4b-it', 'object': 'chat.completion', 'system_fingerprint': None, 'choices': [{'finish_reason': 'stop', 'index': 0, 'message': {'content': "Here's a breakdown of what I see in the image you sent—it's a hospital discharge summary:\n\n**Patient Information:**\n\n*   **Hospital Name:** (Not visible due to handwriting)\n*   **Hospital Address:** (Not visible due to handwriting)\n*   **Patient Name:** (Not visible due to handwriting)\n*   **Patient Address:** (Not visible due to handwriting)\n*   **PMJAY Case ID:** AR - 287\n*   **IPD Number (Free text):** AR - 287\n*   **Package Booked:** Routine Care\n*   **Age:** 60 years\n*   **Sex:** M (Male)\n*   **Patient Contact No.:** (Not visible due to handwriting)\n*   **Registration No.:** (Not visible due to handwriting)\n\n**Treatment Details:**\n\n*   **Treating Consultant's Name:** Dr.  (Handwriting obscured)\n*   **Treating Consultant's C

In [6]:
# =========================
# 3. OUTPUT SCHEMAS
# =========================
# These are the exact expected keys per package, based on the provided examples.

PACKAGE_SCHEMAS = {
    "MG064A": [
        "case_id", "link", "procedure_code", "page_number",
        "clinical_notes", "cbc_hb_report", "indoor_case",
        "treatment_details", "post_hb_report", "discharge_summary",
        "severe_anemia", "common_signs", "significant_signs",
        "life_threatening_signs", "extra_document", "document_rank"
    ],
    "SG039C": [
        "case_id", "S3_link/DocumentName", "procedure_code", "page_number",
        "clinical_notes", "usg_report", "lft_report", "operative_notes",
        "pre_anesthesia", "discharge_summary", "photo_evidence",
        "histopathology", "clinical_condition", "usg_calculi",
        "pain_present", "previous_surgery", "extra_document", "document_rank"
    ],
    "MG006A": [
        "case_id", "S3_link/DocumentName", "procedure_code", "page_number",
        "clinical_notes", "investigation_pre", "pre_date", "vitals_treatment",
        "investigation_post", "post_date", "discharge_summary", "poor_quality",
        "fever", "symptoms", "extra_document", "document_rank"
    ],
    "SB039A": [
        "case_id", "link", "procedure_code", "page_number",
        "clinical_notes", "xray_ct_knee", "indoor_case", "operative_notes",
        "implant_invoice", "post_op_photo", "post_op_xray", "discharge_summary",
        "doa", "dod", "arthritis_type", "post_op_implant_present",
        "age_valid", "extra_document", "document_rank"
    ],
}

KEY_ALIASES = {
    "S3_link": "link",
    "s3_link": "link",
    "S3_link/DocumentName": "link",
}

In [7]:
# =========================
# 4. DATA MODELS
# =========================

@dataclass
class OCRLine:
    text: str
    bbox: Optional[List[int]] = None
    confidence: Optional[float] = None

@dataclass
class PageResult:
    case_id: str
    file_name: str
    page_number: int
    extracted_text: str = ""
    ocr_lines: List[OCRLine] = field(default_factory=list)
    doc_type: str = "unknown"
    doc_type_confidence: float = 0.0
    visual_tags: Dict[str, Any] = field(default_factory=dict)
    entities: Dict[str, Any] = field(default_factory=dict)
    quality: Dict[str, Any] = field(default_factory=dict)
    output_row: Dict[str, Any] = field(default_factory=dict)
    evidence: Dict[str, Any] = field(default_factory=dict)

@dataclass
class TimelineEvent:
    sequence: int
    event_type: str
    date: Optional[str]
    source_document: str
    temporal_validity: str
    evidence: Dict[str, Any] = field(default_factory=dict)

@dataclass
class ClaimDecision:
    case_id: str
    package_code: str
    decision: str
    confidence: float
    reasons: List[str]
    missing_documents: List[str] = field(default_factory=list)
    rule_flags: List[str] = field(default_factory=list)
    timeline_flags: List[str] = field(default_factory=list)

## Recommended pipeline stages

1. **Ingest** claim files  
2. **Split** PDFs/images into pages  
3. **OCR** each page  
4. **Layout / document type classification**  
5. **Visual cue detection** (stamp/signature/QR/barcode/implant sticker/photo evidence)  
6. **Entity extraction** (dates, diagnosis, procedure, age, amounts, etc.)  
7. **Package-specific row creation**  
8. **Timeline construction**  
9. **Rules engine**  
10. **Explainable decisioning**

In [8]:
import os 

os.listdir()

['nha_ps1_skeletal_notebook_main.ipynb',
 '.virtual_documents',
 '.local',
 '.ivy2',
 '.config',
 '.R',
 '.EasyOCR',
 '.cache',
 'c110a5f8-6e79-43bd-bd7a-979677354958',
 'nha_ps1_skeletal_notebook_main_dip.ipynb',
 'v2.ipynb',
 'nha_client.py',
 '.ipynb_checkpoints',
 'outputs',
 '.jupyter',
 '.ipython',
 'lost+found',
 '.julia',
 '__pycache__']

In [9]:
# =========================
# 1. INGEST CLAIM FILES
# =========================
from pathlib import Path
from typing import List

def iter_case_files(case_dir: Path) -> List[Path]:
    """
    Discover all supported files inside a single case directory.

    Expected behavior:
    - recursively scan the case folder
    - keep only files with supported claim-document extensions
    - return a deterministic, sorted list of paths
    - preserve raw file paths exactly as submitted
    """
    folder = Path(case_dir)

    files = [
        file
        for file in folder.rglob("*")
        if file.is_file() and file.suffix.lower() in SUPPORTED_EXTENSIONS
    ]

    return sorted(files)

def discover_cases(data_root: Path) -> Dict[str, List[Path]]:
    '''
    Build the input mapping from case identifier to its associated files.

    Expected behavior when implemented:
    - scan the root input directory
    - treat each child case folder as one claim/case
    - collect supported files for each case using iter_case_files(...)
    - return a dictionary of the form:
      {case_id: [file_path_1, file_path_2, ...]}
    '''
    return {folder : iter_case_files(f'{DATA_ROOT}/{folder}') for folder in os.listdir(f'{DATA_ROOT}')}

In [10]:
! pip install easyocr

In [11]:
import easyocr

reader = easyocr.Reader(['en'])

2026-04-29 19:42:43,274 - WARNING - Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


In [12]:
# =========================
# 2. SPLIT PDFS/IMAGES INTO PAGES
# =========================
import fitz  # PyMuPDF
from PIL import Image
from pathlib import Path
from typing import List, Dict, Any
import io
import easyocr
import numpy as np

reader = easyocr.Reader(['en'])
SUPPORTED_EXTENSIONS = {".pdf", ".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

def extract_pages(file_path: Path) -> List[Dict[str, Any]]:
    file_path = Path(file_path)
    ext = file_path.suffix.lower()
    file_name = file_path.name
    pages = []

    if ext == ".pdf":
        doc = fitz.open(file_path)
        for i, page in enumerate(doc, start=1):
            mat = fitz.Matrix(2.0, 2.0)  # 2x zoom = ~144 DPI, good for OCR
            pix = page.get_pixmap(matrix=mat)
            image = Image.open(io.BytesIO(pix.tobytes("png")))
            pages.append({
                "page_number": i,
                "image": image,
                "file_name": file_name
            })
        doc.close()

    elif ext in {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}:
        image = Image.open(file_path)
        # Handle multi-page TIFFs
        if ext in {".tif", ".tiff"}:
            for i in range(image.n_frames):
                image.seek(i)
                pages.append({
                    "page_number": i + 1,
                    "image": image.copy(),
                    "file_name": file_name
                })
        else:
            pages.append({
                "page_number": 1,
                "image": image,
                "file_name": file_name
            })

    else:
        raise ValueError(f"Unsupported file format: {ext}. Supported: {SUPPORTED_EXTENSIONS}")

    return pages

import json

def ocr_pages(file_path: Path) -> List[Dict[str, Any]]:
    pages = extract_pages(file_path)
    results = []

    for page in pages:
        # Step 1: EasyOCR
        image_np = np.array(page["image"])
        ocr_result = reader.readtext(image_np)
        ocr_lines = [
            {"bbox": bbox, "text": text, "confidence": round(conf, 4)}
            for bbox, text, conf in ocr_result
        ]
        raw_ocr_text = "\n".join([line["text"] for line in ocr_lines])

        # Step 2: Gemma 3 12B — text + visual cues in one call
        buffer = io.BytesIO()
        page["image"].save(buffer, format="JPEG")
        image_base64 = base64.b64encode(buffer.getvalue()).decode("utf-8")
        data_url = f"data:image/jpeg;base64,{image_base64}"

        response = nc.completion(
            model="Gemma 3 4B",
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "image_url", "image_url": {"url": data_url}},
                        {"type": "text", "text": f"""
You are analyzing a medical insurance claim document.

An OCR engine has already extracted the following raw text:
<ocr_extracted>
{raw_ocr_text}
</ocr_extracted>

Analyze the image carefully and return a JSON object with this exact structure:
{{
    "corrected_text": "<full corrected and complete text from the document>",
    "visual_cues": {{
        "has_stamp": true or false,
        "stamp_description": "<color, shape, any text visible in stamp or null>",
        "has_signature": true or false,
        "signature_location": "<top/bottom/middle or null>",
        "has_qr_code": true or false,
        "qr_content": "<decoded content or null>",
        "has_barcode": true or false,
        "barcode_content": "<decoded content or null>",
        "has_implant_sticker": true or false,
        "implant_sticker_text": "<text on sticker or null>",
        "has_photo_evidence": true or false,
        "photo_description": "<what the photo shows or null>",
        "has_letterhead": true or false,
        "has_table": true or false,
        "has_handwriting": true or false,
        "additional_cues": "<any other notable visual elements or null>"
    }},
    "ocr_corrections": [
        {{"original": "<what OCR said>", "corrected": "<what it should be>"}}
    ],
    "missing_from_ocr": ["<text visible in image but missing from OCR>"]
}}

Return ONLY the JSON, no explanation, no markdown backticks.
                        """},
                    ],
                }
            ],
            metadata={"problem_statement": 1}
        )

        # Step 3: Parse JSON response
        raw_response = response["choices"][0]["message"]["content"]
        print(raw_response)
        try:
            parsed = json.loads(raw_response)
        except json.JSONDecodeError:
            # Fallback: strip accidental markdown fences
            clean = raw_response.replace("```json", "").replace("```", "").strip()
            parsed = json.loads(clean)

        results.append({
            "page_number": page["page_number"],
            "file_name": page["file_name"],
            "raw_ocr_lines": ocr_lines,
            "raw_ocr_text": raw_ocr_text,
            "corrected_text": parsed.get("corrected_text", ""),
            "visual_cues": parsed.get("visual_cues", {}),
            "ocr_corrections": parsed.get("ocr_corrections", []),
            "missing_from_ocr": parsed.get("missing_from_ocr", []),
        })

    return results

2026-04-29 19:42:44,865 - WARNING - Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


In [14]:
# =========================
# 3. OCR EACH PAGE
# =========================

# def run_ocr(page_image: Any) -> Tuple[str, List[OCRLine]]:
#     image_np = np.array(page_image)
#     ocr_result = reader.readtext(image_np)

#     lines = []
#     for bbox, text, confidence in ocr_result:
#         # bbox is [[x1,y1],[x2,y2],[x3,y3],[x4,y4]], flatten to [x1,y1,x2,y2,x3,y3,x4,y4]
#         flat_bbox = [int(coord) for point in bbox for coord in point]
#         lines.append(OCRLine(
#             text=text,
#             bbox=flat_bbox,
#             confidence=round(confidence, 4)
#         ))

#     full_text = " ".join(line.text for line in lines)

#     return full_text, lines


In [15]:
! pip install opencv-python

In [16]:
# =========================
# 4. LAYOUT / DOCUMENT TYPE CLASSIFIER
# =========================
import cv2
import numpy as np
from typing import Any, Dict

def estimate_page_quality(page_image: Any, extracted_text: str) -> Dict[str, Any]:
    img_np = np.array(page_image.convert("RGB"))
    gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    h, w = gray.shape

    # --- Blur (Laplacian variance) ---
    blur_score = cv2.Laplacian(gray, cv2.CV_64F).var()
    is_blurry = blur_score < 100

    # --- Contrast (std of pixel intensities) ---
    contrast_score = float(gray.std())
    is_low_contrast = contrast_score < 30

    # --- Noise (diff between raw and gaussian-smoothed) ---
    smoothed = cv2.GaussianBlur(gray, (5, 5), 0)
    noise_score = float(np.abs(gray.astype(np.float32) - smoothed).mean())
    is_noisy = noise_score > 10

    # --- Text density ---
    char_count = len(extracted_text.strip())
    pixel_area = h * w
    text_density = char_count / pixel_area
    is_low_text_density = text_density < 1e-4

    # --- Skew (via edges + Hough lines) ---
    edges = cv2.Canny(gray, 50, 150, apertureSize=3)
    lines = cv2.HoughLines(edges, 1, np.pi / 180, threshold=100)
    skew_angle = 0.0
    if lines is not None:
        angles = [np.degrees(line[0][1]) - 90 for line in lines]
        skew_angle = float(np.median(angles))
    is_skewed = abs(skew_angle) > 5

    # --- Overall usability ---
    is_usable = not (is_blurry and is_low_contrast and is_low_text_density)

    return {
        "is_usable": is_usable,
        "blur_score": round(blur_score, 2),
        "is_blurry": is_blurry,
        "contrast_score": round(contrast_score, 2),
        "is_low_contrast": is_low_contrast,
        "noise_score": round(noise_score, 2),
        "is_noisy": is_noisy,
        "text_density": round(text_density, 6),
        "is_low_text_density": is_low_text_density,
        "skew_angle_deg": round(skew_angle, 2),
        "is_skewed": is_skewed,
    }

In [17]:
estimate_page_quality(pages[0]['image'] , full_text)

NameError: name 'pages' is not defined

In [18]:
! pip install pyzbar

In [19]:
# import cv2
# import numpy as np
# from pyzbar import pyzbar
# from typing import Any, Dict

# def detect_visual_elements(page_image: Any) -> Dict[str, Any]:
#     img_np = np.array(page_image.convert("RGB"))
#     gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
#     h, w = gray.shape

#     # --- QR Code ---
#     qr_detector = cv2.QRCodeDetector()
#     qr_data, _, _ = qr_detector.detectAndDecode(gray)
#     has_qr = bool(qr_data)
#     qr_content = qr_data if has_qr else None

#     # --- Barcode (pyzbar handles most 1D/2D formats) ---
#     barcodes = pyzbar.decode(img_np)
#     has_barcode = len(barcodes) > 0
#     barcode_values = [b.data.decode("utf-8") for b in barcodes] if has_barcode else []

#     # --- Stamp (circular/elliptical contours, often colored) ---
#     hsv = cv2.cvtColor(img_np, cv2.COLOR_RGB2HSV)
#     # Look for red/blue/purple hues typical of stamps
#     red_mask = cv2.inRange(hsv, (0, 50, 50), (10, 255, 255)) | \
#                cv2.inRange(hsv, (160, 50, 50), (180, 255, 255))
#     blue_mask = cv2.inRange(hsv, (100, 50, 50), (140, 255, 255))
#     stamp_mask = cv2.bitwise_or(red_mask, blue_mask)
#     stamp_contours, _ = cv2.findContours(stamp_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
#     has_stamp = any(
#         cv2.contourArea(c) > 500
#         for c in stamp_contours
#     )

#     # --- Signature (bottom 30% of page, irregular dark strokes) ---
#     bottom_region = gray[int(h * 0.7):, :]
#     _, thresh = cv2.threshold(bottom_region, 127, 255, cv2.THRESH_BINARY_INV)
#     sig_contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
#     # Signatures tend to be wide, irregular, and not too large
#     sig_candidates = [
#         c for c in sig_contours
#         if 200 < cv2.contourArea(c) < 15000
#         and cv2.boundingRect(c)[2] > cv2.boundingRect(c)[3]  # wider than tall
#     ]
#     has_signature = len(sig_candidates) >= 2  # multiple strokes = likely signature

#     # --- Implant Sticker (small high-contrast rectangular region) ---
#     edges = cv2.Canny(gray, 50, 150)
#     sticker_contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
#     has_implant_sticker = False
#     for c in sticker_contours:
#         x, y, cw, ch = cv2.boundingRect(c)
#         aspect = cw / ch if ch > 0 else 0
#         area = cw * ch
#         # Implant stickers are typically small rectangles
#         if 1000 < area < 50000 and 1.5 < aspect < 5.0:
#             roi = gray[y:y+ch, x:x+cw]
#             if roi.std() > 40:  # high contrast inside = dense text/print
#                 has_implant_sticker = True
#                 break

#     # --- Photo Evidence (high color saturation region) ---
#     saturation = hsv[:, :, 1]
#     mean_saturation = float(saturation.mean())
#     has_photo_evidence = mean_saturation > 40  # documents are mostly grayscale

#     return {
#         "has_qr_code": has_qr,
#         "qr_content": qr_content,
#         "has_barcode": has_barcode,
#         "barcode_values": barcode_values,
#         "has_stamp": has_stamp,
#         "has_signature": has_signature,
#         "has_implant_sticker": has_implant_sticker,
#         "has_photo_evidence": has_photo_evidence,
#         "mean_saturation": round(mean_saturation, 2),
#     }

In [20]:
# =========================
# ENTITY EXTRACTION HELPERS
# =========================

DATE_PATTERNS = [
    r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b",
    r"\b\d{1,2}-[A-Za-z]{3}-\d{2,4}\b",
    r"\b\d{1,2}\s+[A-Za-z]{3,9}\s+\d{2,4}\b",
]

import re

def find_dates(text: str) -> List[str]:
    found = []
    for pattern in DATE_PATTERNS:
        matches = re.findall(pattern, text)
        found.extend(matches)
    # Deduplicate while preserving order
    seen = set()
    return [d for d in found if not (d in seen or seen.add(d))]

def find_age(text: str) -> Optional[int]:
    patterns = [
        r"\b(\d{1,3})\s*(?:years?|yrs?|Y/O|yo)\b",
        r"\bage[:\s]+(\d{1,3})\b",
        r"\b(\d{1,3})\s*(?:M|F|Male|Female)\b",  # e.g. "45 M"
    ]
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            age = int(match.group(1))
            if 0 < age < 120:  # sanity check
                return age
    return None

def contains_any(text: str, keywords: List[str]) -> int:
    normalized_text = text.lower()
    return 1 if any(kw.lower() in normalized_text for kw in keywords) else 0

In [21]:
# =========================
# 6. ENTITY EXTRACTION
# =========================

DOCUMENT_TYPES = [
    "clinical_notes",
    "cbc_hb_report",
    "indoor_case",
    "treatment_details",
    "post_hb_report",
    "discharge_summary",
    "usg_report",
    "lft_report",
    "operative_notes",
    "pre_anesthesia",
    "histopathology",
    "xray_ct_knee",
    "investigation_pre",
    "investigation_post",
    "vitals_treatment",
    "implant_invoice",
    "post_op_photo",
    "post_op_xray",
    "photo_evidence",
    "extra_document",
]

def prepare_for_classifier(page: Dict[str, Any]) -> Tuple[str, Dict[str, Any]]:
    """
    Prepare the output of ocr_pages for the classify_document_type function.
    Enriches the extracted text with missing OCR text and corrections context.
    """

    # Start with corrected text
    text_parts = [page.get("corrected_text", "")]

    # Append anything the OCR missed
    missing = page.get("missing_from_ocr", [])
    if missing:
        text_parts.append("Additional text found in image:")
        text_parts.extend(missing)

    # Append OCR corrections as context
    corrections = page.get("ocr_corrections", [])
    if corrections:
        text_parts.append("OCR corrections made:")
        for c in corrections:
            text_parts.append(f"  {c['original']} → {c['corrected']}")

    enriched_text = "\n".join(text_parts)

    return enriched_text, page.get("visual_cues", {})

def classify_document_type(extracted_text: str, visual_tags: Dict[str, Any]) -> Dict[str, Any]:
    
    doc_types_str = "\n".join(f"- {dt}" for dt in DOCUMENT_TYPES)

    response = nc.completion(
        model="Gemma 3 4B",
        messages=[
            {
                "role": "user",
                "content": f"""
You are classifying a medical insurance claim document page.

Here is the extracted text from the page:
<extracted_text>
{extracted_text}
</extracted_text>

Here are the detected visual cues:
<visual_cues>
{json.dumps(visual_tags, indent=2)}
</visual_cues>

Classify this document into EXACTLY ONE of these types:
{doc_types_str}

Return ONLY a JSON object with this exact structure:
{{
    "document_type": "<one of the types above>",
    "confidence": <float between 0.0 and 1.0>,
    "reasoning": "<one line explanation>"
}}

If you are unsure, use "extra_document" with a low confidence score.
Return ONLY the JSON, no markdown, no explanation.
                """
            }
        ],
        metadata={"problem_statement": 1}
    )

    raw = response["choices"][0]["message"]["content"]
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        clean = raw.replace("```json", "").replace("```", "").strip()
        parsed = json.loads(clean)

    doc_type = parsed.get("document_type", "extra_document")
    confidence = parsed.get("confidence", 0.0)

    if doc_type not in DOCUMENT_TYPES:
        doc_type = "extra_document"
        confidence = 0.0

    # Run entity extraction helpers on the same text
    dates = find_dates(extracted_text)
    age = find_age(extracted_text)
    print(response)

    return {
        "document_type": doc_type,
        "confidence": confidence,
        "reasoning": parsed.get("reasoning", ""),
        "dates": dates,
        "age": age,
    }   

In [22]:
# =========================
# 7. PAGE-TO-ROW MAPPING
# =========================

def populate_row_for_package(
    package_code: str,
    page_result: PageResult,
) -> Dict[str, Any]:
    """
    Create and populate a single output row for one page, based on the assigned package code and the intermediate page-level analysis result.

    Intended responsibilities:
    - Initialize an output row using the case ID, file name, page number, and package code.
    - Map detected document type to the corresponding package-specific presence field when applicable.
    - Populate package-specific clinical, procedural, temporal, and visual fields using extracted text, detected visual tags, and quality signals.
    - Apply package-specific heuristics or rules for values such as clinical condition flags, symptom flags, dates, implant evidence, age validation, and other STG-relevant attributes.
    - Mark whether the page belongs to an extra/non-required document.
    - Assign a document rank based on page role in the episode timeline, or assign rank 99 for extra documents.
    - Return the final row as a dictionary matching the required output schema.

    Notes for participants:
    - Keep the final keys and output structure exactly aligned with the expected evaluation format.
    - Replace starter heuristics with robust logic driven by OCR, document classification, image understanding, and STG-aware rules.
    - Ensure date extraction and package-specific logic remain explainable and reproducible.
    """
    pass

In [23]:
# =========================
# PACKAGE ROW INITIALIZERS
# =========================

def normalize_output_key(key: str) -> str:
    '''
    Resolve any key aliases so internal logic and required output schema stay aligned.

    Expected behavior when implemented:
    - map alternate/internal key names to the exact organizer-facing keys
    - preserve the strict output format required for evaluation
    '''
    pass

def initialize_output_row(package_code: str, case_id: str, file_name: str, page_number: int) -> Dict[str, Any]:
    '''
    Create an empty page-level output row for a given package code.

    Expected behavior when implemented:
    - initialize all required keys for the selected package schema
    - preserve exact key names and key order
    - fill package metadata such as case id, file name, procedure code, and page number
    - initialize nullable date fields correctly where applicable
    '''
    pass


In [24]:
# =========================
# PAGE-TO-ROW MAPPING
# =========================

def populate_row_for_package(
    package_code: str,
    page_result: PageResult,
) -> Dict[str, Any]:
    '''
    Convert a processed page into one exact output row for the assigned package.

    Expected behavior when implemented:
    - map document-type predictions into presence/absence fields
    - populate package-specific clinical, visual, and temporal fields
    - identify extra documents where applicable
    - assign the appropriate document rank while keeping the exact output format
    '''
    pass


In [25]:
# =========================
# DOCUMENT RANKING
# =========================

RANK_MAP = {
    "MG064A": {
        "clinical_notes": 1,
        "cbc_hb_report": 2,
        "indoor_case": 2,
        "treatment_details": 3,
        "post_hb_report": 4,
        "discharge_summary": 5,
    },
    "SG039C": {
        "clinical_notes": 1,
        "usg_report": 2,
        "lft_report": 3,
        "pre_anesthesia": 4,
        "operative_notes": 5,
        "discharge_summary": 5,
        "histopathology": 6,
        "photo_evidence": 6,
    },
    "MG006A": {
        "clinical_notes": 1,
        "investigation_pre": 2,
        "vitals_treatment": 3,
        "investigation_post": 4,
        "discharge_summary": 5,
    },
    "SB039A": {
        "clinical_notes": 1,
        "xray_ct_knee": 2,
        "indoor_case": 3,
        "operative_notes": 4,
        "implant_invoice": 5,
        "post_op_photo": 6,
        "post_op_xray": 6,
        "discharge_summary": 7,
    }
}

def infer_document_rank(package_code: str, row: Dict[str, Any], doc_type: str) -> Optional[int]:
    '''
    Determine the page/document position in the expected clinical timeline.

    Expected behavior when implemented:
    - assign the package-specific rank based on document type and context
    - return rank 99 for extra documents
    - keep the same rank across all pages belonging to the same logical document where required
    '''
    pass


In [26]:
# =========================
# 8. TIMELINE CONSTRUCTION
# =========================

def build_episode_timeline(package_code: str, page_results: List[PageResult]) -> List[TimelineEvent]:
    '''
    Construct a chronological episode timeline from page-level evidence.

    Expected behavior when implemented:
    - extract admission, investigation, procedure, monitoring, and discharge events
    - order them chronologically or by inferred clinical sequence
    - capture source-document provenance for each event
    - mark basic temporal validity such as before procedure / after treatment / valid
    '''
    pass


In [27]:
# =========================
# 9. RULES ENGINE (STARTER)
# =========================

MANDATORY_DOCS = {
    "MG064A": ["clinical_notes", "cbc_hb_report", "indoor_case", "treatment_details", "post_hb_report", "discharge_summary"],
    "SG039C": ["clinical_notes", "usg_report", "lft_report", "operative_notes", "pre_anesthesia", "discharge_summary", "photo_evidence", "histopathology"],
    "MG006A": ["clinical_notes", "investigation_pre", "vitals_treatment", "investigation_post", "discharge_summary"],
    "SB039A": ["clinical_notes", "xray_ct_knee", "indoor_case", "operative_notes", "implant_invoice", "post_op_photo", "post_op_xray", "discharge_summary"],
}

def aggregate_case_rows(rows: List[Dict[str, Any]]) -> Dict[str, Any]:
    '''
    Collapse page-level output rows into case-level evidence signals.

    Expected behavior when implemented:
    - combine per-page binary flags into case-level presence/absence indicators
    - preserve useful date/metadata fields for downstream rules evaluation
    - make the result suitable for package-level adjudication logic
    '''
    pass

def run_rules_engine(case_id: str, package_code: str, rows: List[Dict[str, Any]], timeline: List[TimelineEvent]) -> ClaimDecision:
    '''
    Evaluate the case against package-specific STG and policy rules.

    Expected behavior when implemented:
    - check mandatory-document completeness
    - check package-specific clinical and temporal plausibility
    - generate Pass / Conditional / Fail with reasons and confidence
    - preserve explainability through explicit flags and missing-document lists
    '''
    pass


In [28]:
# =========================
# 10. EXPLAINABLE DECISIONING
# =========================

def build_human_readable_summary(package_code: str, page_results: List[PageResult], decision: ClaimDecision) -> pd.DataFrame:
    '''
    Build the reviewer-facing summary table for the case.

    Expected behavior when implemented:
    - produce a page/document-level summary aligned with the sample output style
    - show document classification, evidence usage, and rule-related notes
    - remain easy for PPD/CPD reviewers to read
    '''
    pass

def build_timeline_df(timeline: List[TimelineEvent]) -> pd.DataFrame:
    '''
    Convert structured timeline events into a tabular reviewer-friendly format.

    Expected behavior when implemented:
    - preserve sequence, event type, date, source document, and temporal validity
    - produce a DataFrame aligned with the expected timeline output structure
    '''
    pass


In [29]:
# =========================
# CORE PIPELINE DRIVER
# =========================

def process_case(case_id: str, files: List[Path], package_code: str) -> Dict[str, Any]:
    '''
    Run the full page-level and case-level pipeline for a single claim case.

    Expected behavior when implemented:
    - ingest files and split them into pages
    - run OCR, quality checks, visual detection, and document classification
    - convert page evidence into exact output rows
    - build the episode timeline, run rules, and prepare reviewer-facing outputs
    - return a dictionary containing both strict evaluation outputs and readable summaries
    '''
    pass


In [30]:
# =========================
# BATCH RUNNER
# =========================

def run_batch(data_root: Path, package_code_lookup: Optional[Dict[str, str]] = None) -> Dict[str, Any]:
    '''
    Execute the claim-validation pipeline across multiple cases.

    Expected behavior when implemented:
    - discover available cases from the input root
    - resolve the package code for each case
    - call process_case(...) for each valid case
    - collect all outputs into a single batch result dictionary
    '''
    pass


In [31]:
# =========================
# DEMO WITH THE PROVIDED EXAMPLE JSON STRUCTURES
# =========================

EXAMPLE_JSON_PATHS = {
    "SG039C": "data/SG039C_Cholecystectomy.json",
    "SB039A": "data/SB039A_Knee_Replacement.json",
    "MG064A": "data/MG064A_Anemia.json",
    "MG006A": "data/MG006A_Fever.json",
}

def load_example_jsons() -> Dict[str, List[Dict[str, Any]]]:
    out = {}
    for pkg, path in EXAMPLE_JSON_PATHS.items():
        if os.path.exists(path):
            with open(path, "r", encoding="utf-8") as f:
                out[pkg] = json.load(f)
    return out

example_jsons = load_example_jsons()
{k: len(v) for k, v in example_jsons.items()}

{}

In [32]:
# =========================
# EXPORTERS
# =========================

def export_case_outputs(case_result: Dict[str, Any], output_root: Path = OUTPUT_ROOT) -> None:
    '''
    Write strict outputs and reviewer-facing artifacts to disk.

    Expected behavior when implemented:
    - export the exact package-specific JSON rows for evaluation
    - export the human-readable summary and timeline tables
    - export the explainable decision record for the case
    - preserve the existing output folder structure used by this notebook
    '''
    pass


In [33]:
# =========================
# VALIDATOR FOR EXACT OUTPUT KEYS
# =========================

def validate_output_rows(package_code: str, rows: List[Dict[str, Any]]) -> Tuple[bool, List[str]]:
    expected = PACKAGE_SCHEMAS[package_code]
    issues = []

    for i, row in enumerate(rows):
        row_keys = list(row.keys())
        if row_keys != expected:
            issues.append(
                f"Row {i}: key order / names mismatch.\nExpected: {expected}\nGot:      {row_keys}"
            )
    return len(issues) == 0, issues

In [34]:
# =========================
# EXAMPLE: VALIDATE ORGANIZER JSON SAMPLES
# =========================

for pkg, rows in example_jsons.items():
    ok, issues = validate_output_rows(pkg, rows)
    print(pkg, "->", "VALID" if ok else "INVALID")
    if issues:
        print("\n".join(issues[:2]))

## Suggested participant extensions

Participants can improve this scaffold by adding:

- multilingual OCR
- VLM-based page classification
- stamp/signature/implant sticker detection
- table extraction for invoices / vitals
- robust date normalization to `DD-MM-YYYY`
- exact STG rule encoding per package
- evidence spans and bounding boxes
- calibrated confidence scoring
- duplicate / redundant / extra document tagging
- page grouping into document-level clusters

In [35]:
# =========================
# DATE NORMALIZATION UTIL
# =========================

from datetime import datetime

def normalize_date(date_str: Optional[str]) -> Optional[str]:
    if not date_str:
        return None

    candidates = [
        "%d/%m/%y", "%d/%m/%Y",
        "%d-%m-%y", "%d-%m-%Y",
        "%d-%b-%y", "%d-%b-%Y",
        "%d %b %Y", "%d %B %Y",
        "%m/%d/%y", "%m/%d/%Y",  # keep only if your data needs it
    ]

    for fmt in candidates:
        try:
            dt = datetime.strptime(date_str, fmt)
            return dt.strftime("%d-%m-%Y")
        except Exception:
            continue
    return date_str

In [36]:
# =========================
# OPTIONAL: POST-PROCESS DATES INTO REQUIRED FORMAT
# =========================

def normalize_dates_in_rows(package_code: str, rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    date_keys = []
    if package_code == "MG006A":
        date_keys = ["pre_date", "post_date"]
    elif package_code == "SB039A":
        date_keys = ["doa", "dod"]

    normalized = []
    for row in rows:
        r = dict(row)
        for dk in date_keys:
            if dk in r:
                r[dk] = normalize_date(r.get(dk))
        normalized.append(r)
    return normalized

In [37]:
# =========================
# DATE NORMALIZATION UTIL
# =========================

from datetime import datetime

def normalize_date(date_str: Optional[str]) -> Optional[str]:
    '''
    Normalize a raw date string into the required organizer output format.

    Expected behavior when implemented:
    - accept multiple common date formats extracted from OCR/report text
    - convert valid dates into a standardized output representation
    - return None when no valid date can be parsed reliably
    '''
    pass


In [38]:
# =========================
# POST-PROCESS DATES INTO REQUIRED FORMAT
# =========================

def normalize_dates_in_rows(package_code: str, rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    '''
    Apply date normalization only to the package-specific date fields.

    Expected behavior when implemented:
    - identify which date keys belong to the current package output schema
    - normalize those fields using normalize_date(...)
    - return rows in the same schema/order expected by evaluation
    '''
    pass

In [39]:
# =========================
# SAMPLE DECISION REPORT RENDERER
# =========================

def render_decision_report(case_result: Dict[str, Any]) -> None:
    '''
    Render a simple reviewer-facing decision card in notebook output.

    Expected behavior when implemented:
    - display case id, package code, final decision, confidence, and reasons
    - keep the presentation concise and easy to review during the hackathon
    '''
    pass

In [40]:
from concurrent.futures import ThreadPoolExecutor


def _extract_json_object(text: str) -> Dict[str, Any]:
    """Extract the last valid JSON object from a model response."""
    text = text.strip()

    fenced_matches = re.findall(r"```json\s*(.*?)\s*```", text, flags=re.DOTALL | re.IGNORECASE)
    for candidate in reversed(fenced_matches):
        try:
            return json.loads(candidate.strip())
        except json.JSONDecodeError:
            pass

    decoder = json.JSONDecoder()
    brace_candidates = [index for index, char in enumerate(text) if char == "{"]
    for start in reversed(brace_candidates):
        try:
            parsed, _ = decoder.raw_decode(text[start:])
            if isinstance(parsed, dict):
                return parsed
        except json.JSONDecodeError:
            continue

    raise json.JSONDecodeError("No valid JSON object found", text, 0)


def _run_ocr_completion(page: Dict[str, Any], raw_ocr_text: str, data_url: str) -> Dict[str, Any]:
    response = nc.completion(
        model="Gemma 3 4B",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "image_url", "image_url": {"url": data_url}},
                    {"type": "text", "text": f"""
You are analyzing a medical insurance claim document.

An OCR engine has already extracted the following raw text:
<ocr_extracted>
{raw_ocr_text}
</ocr_extracted>

Analyze the image carefully and return a JSON object with this exact structure:
{{
    "corrected_text": "<full corrected and complete text from the document>",
    "visual_cues": {{
        "has_stamp": true or false,
        "stamp_description": "<color, shape, any text visible in stamp or null>",
        "has_signature": true or false,
        "signature_location": "<top/bottom/middle or null>",
        "has_qr_code": true or false,
        "qr_content": "<decoded content or null>",
        "has_barcode": true or false,
        "barcode_content": "<decoded content or null>",
        "has_implant_sticker": true or false,
        "implant_sticker_text": "<text on sticker or null>",
        "has_photo_evidence": true or false,
        "photo_description": "<what the photo shows or null>",
        "has_letterhead": true or false,
        "has_table": true or false,
        "has_handwriting": true or false,
        "additional_cues": "<any other notable visual elements or null>"
    }},
    "ocr_corrections": [
        {{"original": "<what OCR said>", "corrected": "<what it should be>"}}
    ],
    "missing_from_ocr": ["<text visible in image but missing from OCR>"]
}}

Return ONLY the JSON, no explanation, no markdown backticks.
                        """},
                ],
            }
        ],
        metadata={"problem_statement": 1}
    )

    raw_response = response["choices"][0]["message"]["content"]
    parsed = _extract_json_object(raw_response)

    return {
        "page_number": page["page_number"],
        "file_name": page["file_name"],
        "raw_ocr_lines": page["raw_ocr_lines"],
        "raw_ocr_text": raw_ocr_text,
        "corrected_text": parsed.get("corrected_text", ""),
        "visual_cues": parsed.get("visual_cues", {}),
        "ocr_corrections": parsed.get("ocr_corrections", []),
        "missing_from_ocr": parsed.get("missing_from_ocr", []),
    }


def ocr_pages(file_path: Path) -> List[Dict[str, Any]]:
    pages = extract_pages(file_path)
    prepared_pages = []

    for page in pages:
        image_np = np.array(page["image"])
        ocr_result = reader.readtext(image_np)
        ocr_lines = [
            {"bbox": bbox, "text": text, "confidence": round(conf, 4)}
            for bbox, text, conf in ocr_result
        ]
        raw_ocr_text = "\n".join([line["text"] for line in ocr_lines])

        buffer = io.BytesIO()
        page["image"].save(buffer, format="JPEG")
        image_base64 = base64.b64encode(buffer.getvalue()).decode("utf-8")
        data_url = f"data:image/jpeg;base64,{image_base64}"

        prepared_pages.append({
            "page_number": page["page_number"],
            "file_name": page["file_name"],
            "raw_ocr_lines": ocr_lines,
            "raw_ocr_text": raw_ocr_text,
            "data_url": data_url,
        })

    if not prepared_pages:
        return []

    max_workers = min(4, len(prepared_pages))
    results = [None] * len(prepared_pages)

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_index = {
            executor.submit(
                _run_ocr_completion,
                page,
                page["raw_ocr_text"],
                page["data_url"],
            ): index
            for index, page in enumerate(prepared_pages)
        }

        for future, index in future_to_index.items():
            results[index] = future.result()

    return results

In [41]:
# =========================
# 11. COMPLETED-FUNCTIONS BASELINE RUNNER
# =========================

from datetime import datetime
import io
import json
from typing import Any, Dict, List, Optional, Tuple

COMPLETED_RUNNER_OUTPUT_ROOT = OUTPUT_ROOT / "completed_functions_runner"
COMPLETED_RUNNER_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

LOCAL_RANK_HINTS = {
    "MG064A": {
        "clinical_notes": 1,
        "cbc_hb_report": 2,
        "indoor_case": 2,
        "treatment_details": 3,
        "post_hb_report": 4,
        "discharge_summary": 5,
    },
    "SG039C": {
        "clinical_notes": 1,
        "usg_report": 2,
        "lft_report": 3,
        "pre_anesthesia": 4,
        "operative_notes": 5,
        "discharge_summary": 5,
        "histopathology": 6,
        "photo_evidence": 6,
    },
    "MG006A": {
        "clinical_notes": 1,
        "investigation_pre": 2,
        "vitals_treatment": 3,
        "investigation_post": 4,
        "discharge_summary": 5,
    },
    "SB039A": {
        "clinical_notes": 1,
        "xray_ct_knee": 2,
        "indoor_case": 3,
        "operative_notes": 4,
        "implant_invoice": 5,
        "post_op_photo": 6,
        "post_op_xray": 6,
        "discharge_summary": 7,
    },
}

LOCAL_DOC_KEYWORDS = {
    "MG064A": {
        "clinical_notes": ["clinical notes", "consultation", "history", "examination"],
        "cbc_hb_report": ["cbc", "hb", "hemoglobin", "haemoglobin", "blood report"],
        "indoor_case": ["indoor", "admission", "ipd", "ward"],
        "treatment_details": ["treatment", "medication", "procedure", "therapy"],
        "post_hb_report": ["post hb", "post-hb", "follow up hb", "repeat hb"],
        "discharge_summary": ["discharge summary", "discharge", "final diagnosis"],
    },
    "SG039C": {
        "clinical_notes": ["clinical notes", "consultation", "history"],
        "usg_report": ["usg", "ultrasound", "sonography", "gall bladder"],
        "lft_report": ["lft", "liver function", "bilirubin", "sgot", "sgpt"],
        "operative_notes": ["operative notes", "operation notes", "surgery"],
        "pre_anesthesia": ["pre anesthesia", "pre-anesthesia", "anaesthesia"],
        "discharge_summary": ["discharge summary", "discharge"],
        "histopathology": ["histopathology", "biopsy", "pathology"],
        "photo_evidence": ["photo", "image", "laparoscopic", "surgical site"],
    },
    "MG006A": {
        "clinical_notes": ["clinical notes", "consultation", "history"],
        "investigation_pre": ["investigation", "lab", "cbc", "before"],
        "vitals_treatment": ["vitals", "treatment", "monitoring"],
        "investigation_post": ["investigation", "lab", "after", "repeat"],
        "discharge_summary": ["discharge summary", "discharge"],
    },
    "SB039A": {
        "clinical_notes": ["clinical notes", "consultation", "history"],
        "xray_ct_knee": ["xray", "x-ray", "ct knee", "knee", "radiology"],
        "indoor_case": ["indoor", "admission", "ipd", "ward"],
        "operative_notes": ["operative notes", "operation notes", "surgery"],
        "implant_invoice": ["invoice", "implant", "prosthesis", "hardware"],
        "post_op_photo": ["post op photo", "post-operative photo", "photo"],
        "post_op_xray": ["post op xray", "post-operative xray", "xray"],
        "discharge_summary": ["discharge summary", "discharge"],
    },
}

DATE_FORMATS = [
    "%d/%m/%y",
    "%d/%m/%Y",
    "%d-%m-%y",
    "%d-%m-%Y",
    "%d-%b-%y",
    "%d-%b-%Y",
    "%d %b %Y",
    "%d %B %Y",
    "%m/%d/%y",
    "%m/%d/%Y",
]


def _local_normalize_date(date_str: Optional[str]) -> Optional[str]:
    if not date_str:
        return None

    cleaned = str(date_str).strip()
    for fmt in DATE_FORMATS:
        try:
            return datetime.strptime(cleaned, fmt).strftime("%d-%m-%Y")
        except Exception:
            continue
    return cleaned


def _local_ocr(page_image: Any) -> Tuple[str, List[OCRLine]]:
    image_np = np.array(page_image.convert("RGB"))
    ocr_result = reader.readtext(image_np)

    lines: List[OCRLine] = []
    for bbox, text, confidence in ocr_result:
        flat_bbox = [int(coord) for point in bbox for coord in point]
        lines.append(
            OCRLine(
                text=text,
                bbox=flat_bbox,
                confidence=round(float(confidence), 4),
            )
        )

    full_text = "\n".join(line.text for line in lines)
    return full_text, lines


def _count_keyword_hits(text: str, keywords: List[str]) -> int:
    normalized_text = text.lower()
    return sum(1 for keyword in keywords if keyword.lower() in normalized_text)


def _infer_document_type(package_code: str, extracted_text: str, visual_tags: Dict[str, Any]) -> str:
    normalized_text = extracted_text.lower()
    best_type = "extra_document"
    best_score = 0

    for doc_type, keywords in LOCAL_DOC_KEYWORDS.get(package_code, {}).items():
        score = _count_keyword_hits(normalized_text, keywords)
        if doc_type == "photo_evidence" and visual_tags.get("has_photo_evidence"):
            score += 2
        if doc_type in {"post_op_photo", "photo_evidence"} and visual_tags.get("has_photo_evidence"):
            score += 1
        if doc_type in {"post_op_xray", "xray_ct_knee"} and visual_tags.get("has_barcode"):
            score += 1
        if score > best_score:
            best_type = doc_type
            best_score = score

    if best_score == 0 and visual_tags.get("has_photo_evidence"):
        return "photo_evidence" if package_code == "SG039C" else "post_op_photo"
    return best_type


def _infer_package_flags(package_code: str, doc_type: str, extracted_text: str, visual_tags: Dict[str, Any], quality: Dict[str, Any]) -> Dict[str, Any]:
    text = extracted_text.lower()
    flags: Dict[str, Any] = {}

    if package_code == "MG064A":
        flags["severe_anemia"] = int(any(token in text for token in ["anemia", "anaemia", "low hb", "severe"]))
        flags["common_signs"] = int(any(token in text for token in ["weakness", "fatigue", "pallor", "dizziness", "tiredness"]))
        flags["significant_signs"] = int(any(token in text for token in ["breathless", "tachycardia", "syncope", "transfusion", "bleeding"]))
        flags["life_threatening_signs"] = int(any(token in text for token in ["shock", "icu", "critical", "life threatening", "severe breathlessness"]))
    elif package_code == "SG039C":
        flags["clinical_condition"] = int(any(token in text for token in ["cholecyst", "gall bladder", "biliary", "cholelithiasis"]))
        flags["usg_calculi"] = int(any(token in text for token in ["stone", "calculi", "calculus", "gallstone"]))
        flags["pain_present"] = int(any(token in text for token in ["pain", "colic", "abdomen", "rt hypochondrium"]))
        flags["previous_surgery"] = int(any(token in text for token in ["previous surgery", "h/o surgery", "past surgery", "operated"]))
    elif package_code == "MG006A":
        flags["poor_quality"] = int(not quality.get("is_usable", True))
        flags["fever"] = int("fever" in text)
        flags["symptoms"] = int(any(token in text for token in ["cough", "cold", "weakness", "pain", "ache", "chills", "headache"]))
    elif package_code == "SB039A":
        flags["arthritis_type"] = "osteoarthritis" if any(token in text for token in ["osteoarthritis", "oa", "degenerative", "arthritis"]) else None
        flags["post_op_implant_present"] = int(visual_tags.get("has_implant_sticker", False) or any(token in text for token in ["implant", "prosthesis", "tkr", "replacement"]))

    return flags


def _infer_document_rank(package_code: str, doc_type: str) -> int:
    return int(LOCAL_RANK_HINTS.get(package_code, {}).get(doc_type, 99))


def _build_row(package_code: str, case_id: str, file_name: str, page_number: int, extracted_text: str, visual_tags: Dict[str, Any], quality: Dict[str, Any]) -> Tuple[Dict[str, Any], str, Dict[str, Any]]:
    doc_type = _infer_document_type(package_code, extracted_text, visual_tags)
    dates = find_dates(extracted_text)
    age = find_age(extracted_text)
    normalized_dates = [_local_normalize_date(date_value) for date_value in dates]
    flag_values = _infer_package_flags(package_code, doc_type, extracted_text, visual_tags, quality)

    row: Dict[str, Any] = {}
    schema = PACKAGE_SCHEMAS[package_code]
    for key in schema:
        if key == "case_id":
            row[key] = case_id
        elif key in {"link", "S3_link/DocumentName"}:
            row[key] = str(file_name)
        elif key == "procedure_code":
            row[key] = package_code
        elif key == "page_number":
            row[key] = page_number
        elif key == doc_type:
            row[key] = 1
        elif key == "extra_document":
            row[key] = int(doc_type == "extra_document")
        elif key == "document_rank":
            row[key] = _infer_document_rank(package_code, doc_type)
        elif key in {"pre_date", "post_date", "doa", "dod"}:
            row[key] = normalized_dates[0] if normalized_dates else None
        elif key == "age_valid":
            row[key] = int(age is not None and 45 <= age <= 90)
        elif key in {"clinical_notes", "cbc_hb_report", "indoor_case", "treatment_details", "post_hb_report", "discharge_summary", "usg_report", "lft_report", "operative_notes", "pre_anesthesia", "histopathology", "xray_ct_knee", "investigation_pre", "investigation_post", "vitals_treatment", "implant_invoice", "post_op_photo", "post_op_xray", "photo_evidence"}:
            row[key] = 1 if key == doc_type else 0
        elif key in flag_values:
            row[key] = flag_values[key]
        else:
            row[key] = 0

    if package_code == "MG006A" and len(normalized_dates) > 1:
        row["post_date"] = normalized_dates[1]
    if package_code == "SB039A" and len(normalized_dates) > 1:
        row["dod"] = normalized_dates[1]

    return row, doc_type, {"dates": normalized_dates, "age": age}


def run_completed_functions_runner(data_root: Path = DATA_ROOT) -> Dict[str, Any]:
    cases = discover_cases(data_root)
    if not cases:
        print(f"No cases found under: {data_root.resolve()}")
        return {}

    batch_results: Dict[str, Any] = {}
    all_rows: List[Dict[str, Any]] = []
    summary_rows: List[Dict[str, Any]] = []

    package_lookup = globals().get("PACKAGE_CODE_LOOKUP", {}) or {}

    for case_id, files in cases.items():
        if case_id.startswith("."):
            continue

        package_code = package_lookup.get(case_id)
        if package_code is None and case_id in PACKAGE_CODES:
            package_code = case_id
        if package_code is None:
            helper_file = data_root / case_id / "package_code.txt"
            if helper_file.exists():
                package_code = helper_file.read_text(encoding="utf-8").strip()

        if package_code not in PACKAGE_CODES:
            print(f"Skipping case '{case_id}' because no valid package code was found.")
            continue

        page_results: List[PageResult] = []
        strict_rows: List[Dict[str, Any]] = []

        for file_path in files:
            extracted_pages = extract_pages(file_path)
            llm_pages = ocr_pages(file_path)
            llm_by_page = {page.get("page_number"): page for page in llm_pages}

            for page in extracted_pages:
                page_image = page["image"]
                file_name = page["file_name"]
                page_number = page["page_number"]
                llm_page = llm_by_page.get(page_number, {})

                extracted_text = llm_page.get("corrected_text") or llm_page.get("raw_ocr_text") or ""
                ocr_lines = [
                    OCRLine(
                        text=line.get("text", ""),
                        bbox=[int(v) for point in line.get("bbox", []) for v in (point if isinstance(point, list) else [point])]
                        if line.get("bbox")
                        else None,
                        confidence=line.get("confidence"),
                    )
                    for line in llm_page.get("raw_ocr_lines", [])
                ]
                visual_tags = llm_page.get("visual_cues", {}) or {}
                quality = estimate_page_quality(page_image, extracted_text)
                row, doc_type, entity_info = _build_row(
                    package_code=package_code,
                    case_id=case_id,
                    file_name=file_name,
                    page_number=page_number,
                    extracted_text=extracted_text,
                    visual_tags=visual_tags,
                    quality=quality,
                )

                page_result = PageResult(
                    case_id=case_id,
                    file_name=str(file_name),
                    page_number=page_number,
                    extracted_text=extracted_text,
                    ocr_lines=ocr_lines,
                    doc_type=doc_type,
                    doc_type_confidence=1.0 if doc_type != "extra_document" else 0.25,
                    visual_tags=visual_tags,
                    entities=entity_info,
                    quality=quality,
                    output_row=row,
                    evidence={"visual_tags": visual_tags, "quality": quality},
                )

                page_results.append(page_result)
                strict_rows.append(row)
                all_rows.append(row)
                summary_rows.append(
                    {
                        "case_id": case_id,
                        "package_code": package_code,
                        "file_name": str(file_name),
                        "page_number": page_number,
                        "doc_type": doc_type,
                        "document_rank": row.get("document_rank"),
                        "quality_ok": quality.get("is_usable"),
                        "has_stamp": visual_tags.get("has_stamp"),
                        "has_signature": visual_tags.get("has_signature"),
                        "has_qr_code": visual_tags.get("has_qr_code"),
                        "has_barcode": visual_tags.get("has_barcode"),
                        "dates": entity_info.get("dates", []),
                        "age": entity_info.get("age"),
                    }
                )

        ok, issues = validate_output_rows(package_code, strict_rows)
        case_folder = COMPLETED_RUNNER_OUTPUT_ROOT / case_id
        case_folder.mkdir(parents=True, exist_ok=True)

        rows_path = case_folder / f"{case_id}_{package_code}_rows.json"
        summary_path = case_folder / f"{case_id}_{package_code}_summary.csv"
        pages_path = case_folder / f"{case_id}_{package_code}_pages.json"

        with open(rows_path, "w", encoding="utf-8") as f:
            json.dump(strict_rows, f, indent=2)

        pd.DataFrame(summary_rows[-len(strict_rows):]).to_csv(summary_path, index=False)
        with open(pages_path, "w", encoding="utf-8") as f:
            json.dump([asdict(page_result) for page_result in page_results], f, indent=2, default=str)

        batch_results[case_id] = {
            "case_id": case_id,
            "package_code": package_code,
            "page_results": page_results,
            "strict_rows": strict_rows,
            "validation_ok": ok,
            "validation_issues": issues,
            "output_files": {
                "rows_json": str(rows_path),
                "summary_csv": str(summary_path),
                "pages_json": str(pages_path),
            },
        }

        print(
            f"Case {case_id}: {len(strict_rows)} page row(s), "
            f"validation={'OK' if ok else 'ISSUES'}"
        )
        if issues:
            print(issues[0])

    global COMPLETED_RUNNER_RESULTS, COMPLETED_STRICT_OUTPUTS_DF, COMPLETED_SUMMARY_DF
    COMPLETED_RUNNER_RESULTS = batch_results
    COMPLETED_STRICT_OUTPUTS_DF = pd.DataFrame(all_rows)
    COMPLETED_SUMMARY_DF = pd.DataFrame(summary_rows)

    print(f"Finished processing {len(batch_results)} case(s).")
    if not COMPLETED_STRICT_OUTPUTS_DF.empty:
        print("\nStrict output preview:")
        display(COMPLETED_STRICT_OUTPUTS_DF.head())
    if not COMPLETED_SUMMARY_DF.empty:
        print("\nSummary preview:")
        display(COMPLETED_SUMMARY_DF.head())

    return batch_results


COMPLETED_RUNNER_RESULTS = run_completed_functions_runner(DATA_ROOT)


/opt/conda/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
2026-04-29 19:43:13,052 - INFO - HTTP Request: POST https://aaehackathon.nhaad.in/nhm-proxy/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-29 19:43:23,281 - INFO - HTTP Request: POST https://aaehackathon.nhaad.in/nhm-proxy/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-29 19:43:30,296 - INFO - HTTP Request: POST https://aaehackathon.nhaad.in/nhm-proxy/v1/chat/completions "HTTP/1.1 200 OK"


KeyboardInterrupt: 

In [ ]:
os.listdir(DATA_ROOT)

In [ ]:
COMPLETED_RUNNER_RESULTS